# 03 · Apache Spark – Batch Processing (cleaning, recommendations, cohorts)
**Part 2.1 of the assignment (30 % with Spark SQL).**

Pipeline built in this notebook:

```
raw JSON arrays ──► NDJSON (splittable) ──► explicit schemas ──► CLEAN + VALIDATE + ENRICH ──► Parquet (columnar, partitioned)
                                                                                                    │
                                                    ┌───────────────────────────────────────────────┤
                                                    ▼                                               ▼
                                   Recommendation indicators                          Cohort analysis by registration month
                                   (co-purchase + co-view, lift)                      (spend & retention per months-since-registration)
```
Everything is written to `data/processed/` and reused by notebooks 04 (Spark SQL), 05 (integration) and 06 (charts).

### Why Spark here (and not MongoDB/HBase)?
* The jobs touch **every** session and transaction (self-joins, window functions, pivots) – full-scan, shuffle-heavy work is exactly what a distributed engine is built for.
* Spark is **engine, not store**: it reads files today and Mongo/HBase tomorrow, and scales by adding executors without changing code.

## Step 1 · Start Spark

In [1]:
import sys; sys.path.append("../src")
from config import *
import json, glob, shutil
import pandas as pd
from pyspark.sql import functions as F, types as T, Window

spark = get_spark("03-batch-processing")
print("Spark", spark.version, "| cores:", spark.sparkContext.defaultParallelism, "| shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
pd.set_option("display.max_columns", 40, "display.width", 220)

Spark 4.0.5 | cores: 8 | shuffle partitions: 16


## Step 2 · Convert the raw JSON *arrays* to NDJSON (one record per line)
The generator writes each file as **one giant JSON array**. Spark can only read that with `multiLine=true`, which makes **each file a single, unsplittable record** → one core parses it while the others idle, and the whole file must fit in memory.
Converting once to **NDJSON** (newline-delimited JSON) makes every file splittable and parallel. *(First optimisation – measured in Step 8.)*

In [2]:
NDJSON = PROCESSED / "ndjson"

def array_to_ndjson(src, dst):
    """Stream a JSON array file into a one-object-per-line file."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    data = json.load(open(src))
    with open(dst, "w") as out:
        for rec in data:
            out.write(json.dumps(rec, separators=(",", ":")) + "\n")
    return len(data)

if NDJSON.exists(): shutil.rmtree(NDJSON)
with timer("convert to NDJSON"):
    for name in ["users", "products", "categories", "transactions"]:
        print(f"{name:13s}", f"{array_to_ndjson(RAW / (name + '.json'), NDJSON / (name + '.jsonl')):>9,} records")
    for f in sorted(RAW.glob("sessions_*.json")):
        print(f"{f.stem:13s}", f"{array_to_ndjson(f, NDJSON / 'sessions' / (f.stem + '.jsonl')):>9,} records")

users            10,000 records
products          5,000 records
categories           25 records
transactions     50,000 records
sessions_0      100,000 records
sessions_1      100,000 records
[timer] convert to NDJSON: 21.41s


## Step 3 · Read with **explicit schemas**
Never let Spark *infer* the schema here: 
1. inference needs an extra full pass over the data, and 
2. **`cart_contents` uses product ids as field names** – inference would create a struct with thousands of columns. We declare it as a `MapType` instead.

In [3]:
S, I, D, B = T.StringType(), T.IntegerType(), T.DoubleType(), T.BooleanType()
geo      = T.StructType([T.StructField(n, S) for n in ("city", "state", "country")])
sess_geo = T.StructType([T.StructField(n, S) for n in ("city", "state", "country", "ip_address")])

users_schema = T.StructType([T.StructField("user_id", S), T.StructField("geo_data", geo),
                             T.StructField("registration_date", S), T.StructField("last_active", S)])
categories_schema = T.StructType([T.StructField("category_id", S), T.StructField("name", S),
    T.StructField("subcategories", T.ArrayType(T.StructType([T.StructField("subcategory_id", S), T.StructField("name", S), T.StructField("profit_margin", D)])))])
products_schema = T.StructType([T.StructField("product_id", S), T.StructField("name", S), T.StructField("category_id", S),
    T.StructField("base_price", D), T.StructField("current_stock", I), T.StructField("is_active", B),
    T.StructField("price_history", T.ArrayType(T.StructType([T.StructField("price", D), T.StructField("date", S)]))),
    T.StructField("creation_date", S)])
sessions_schema = T.StructType([T.StructField("session_id", S), T.StructField("user_id", S), T.StructField("start_time", S),
    T.StructField("end_time", S), T.StructField("duration_seconds", I), T.StructField("geo_data", sess_geo),
    T.StructField("device_profile", T.StructType([T.StructField(n, S) for n in ("type", "os", "browser")])),
    T.StructField("viewed_products", T.ArrayType(S)),
    T.StructField("page_views", T.ArrayType(T.StructType([T.StructField("timestamp", S), T.StructField("page_type", S),
        T.StructField("product_id", S), T.StructField("category_id", S), T.StructField("view_duration", I)]))),
    T.StructField("cart_contents", T.MapType(S, T.StructType([T.StructField("quantity", I), T.StructField("price", D)]))),
    T.StructField("conversion_status", S), T.StructField("referrer", S)])
transactions_schema = T.StructType([T.StructField("transaction_id", S), T.StructField("session_id", S), T.StructField("user_id", S),
    T.StructField("timestamp", S),
    T.StructField("items", T.ArrayType(T.StructType([T.StructField("product_id", S), T.StructField("quantity", I),
                                                      T.StructField("unit_price", D), T.StructField("subtotal", D)]))),
    T.StructField("subtotal", D), T.StructField("discount", D), T.StructField("total", D),
    T.StructField("payment_method", S), T.StructField("status", S)])

users_raw        = spark.read.schema(users_schema).json(str(NDJSON / "users.jsonl"))
categories_raw   = spark.read.schema(categories_schema).json(str(NDJSON / "categories.jsonl"))
products_raw     = spark.read.schema(products_schema).json(str(NDJSON / "products.jsonl"))
transactions_raw = spark.read.schema(transactions_schema).json(str(NDJSON / "transactions.jsonl"))
sessions_raw     = spark.read.schema(sessions_schema).json(str(NDJSON / "sessions"))

for n, df in [("users", users_raw), ("categories", categories_raw), ("products", products_raw), ("transactions", transactions_raw), ("sessions", sessions_raw)]:
    print(f"{n:13s} {df.count():>9,} rows | partitions: {df.rdd.getNumPartitions()}")
sessions_raw.printSchema()

users            10,000 rows | partitions: 1
categories           25 rows | partitions: 1
products          5,000 rows | partitions: 1
transactions     50,000 rows | partitions: 5
sessions        200,000 rows | partitions: 8
root
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- start_time: string (nullable = true)
 |-- end_time: string (nullable = true)
 |-- duration_seconds: integer (nullable = true)
 |-- geo_data: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- state: string (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- ip_address: string (nullable = true)
 |-- device_profile: struct (nullable = true)
 |    |-- type: string (nullable = true)
 |    |-- os: string (nullable = true)
 |    |-- browser: string (nullable = true)
 |-- viewed_products: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- page_views: array (nullable = true)
 |    |-- element: struct (containsNull

## Step 4 · Clean, validate and enrich
For each entity: **(1)** remove duplicates / invalid keys, **(2)** parse and standardise types and text, **(3)** handle missing values explicitly, **(4)** add *data-quality flags* instead of silently dropping rows, **(5)** derive analysis-ready columns.

### 4.1 Categories & products

In [4]:
# Categories: one row per category with the average margin of its subcategories (the only margin we can attach to products)
categories_clean = (categories_raw.dropDuplicates(["category_id"]).filter(F.col("category_id").isNotNull())
    .select("category_id", F.trim("name").alias("category_name"),
            F.size("subcategories").alias("n_subcategories"),
            F.round(F.aggregate("subcategories", F.lit(0.0), lambda a, x: a + x["profit_margin"]) / F.size("subcategories"), 4).alias("avg_profit_margin")))

# Sub-category dimension kept separately (for completeness / SQL demos)
subcategories_dim = (categories_raw.select("category_id", F.explode("subcategories").alias("s"))
                     .select("category_id", F.col("s.subcategory_id").alias("subcategory_id"), F.col("s.name").alias("subcategory_name"), F.col("s.profit_margin").alias("profit_margin")))

# Products: price history sorted by date -> current / first price, number of price changes
sorted_hist = F.expr("array_sort(price_history, (l, r) -> case when l.date < r.date then -1 when l.date > r.date then 1 else 0 end)")
products_clean = (products_raw.dropDuplicates(["product_id"]).filter(F.col("product_id").isNotNull())
    .withColumn("hist", sorted_hist)
    .withColumn("product_name", F.trim("name"))
    .withColumn("first_price", F.element_at("hist", 1)["price"])
    .withColumn("current_price", F.coalesce(F.element_at("hist", -1)["price"], F.col("base_price")))
    .withColumn("n_price_changes", F.greatest(F.size("hist") - 1, F.lit(0)))
    .withColumn("price_change_pct", F.round((F.col("current_price") - F.col("first_price")) / F.col("first_price") * 100, 2))
    .withColumn("creation_ts", F.to_timestamp("creation_date"))
    .withColumn("is_out_of_stock", F.col("current_stock") <= 0)
    .withColumn("current_stock", F.coalesce("current_stock", F.lit(0)))
    .withColumn("is_active", F.coalesce("is_active", F.lit(False)))
    .join(F.broadcast(categories_clean), "category_id", "left")             # small dimension -> broadcast join
    .select("product_id", "product_name", "category_id", "category_name", "avg_profit_margin", "base_price", "current_price",
            "first_price", "n_price_changes", "price_change_pct", "current_stock", "is_out_of_stock", "is_active", "creation_ts"))
products_clean.show(5, truncate=40)

+----------+--------------------------------------+-----------+--------------+-----------------+----------+-------------+-----------+---------------+----------------+-------------+---------------+---------+-------------------+
|product_id|                          product_name|category_id| category_name|avg_profit_margin|base_price|current_price|first_price|n_price_changes|price_change_pct|current_stock|is_out_of_stock|is_active|        creation_ts|
+----------+--------------------------------------+-----------+--------------+-----------------+----------+-------------+-----------+---------------+----------------+-------------+---------------+---------+-------------------+
|prod_00000| Synchronized Bi-Directional Help-Desk|    cat_016|Wright-Leblanc|             0.26|     371.3|        371.3|     459.65|              2|          -19.22|          896|          false|     true|2026-05-23 16:00:32|
|prod_00001|          Open-Source Regional Synergy|    cat_009|Skinner-Romero|           0.2

### 4.2 Users

In [5]:
users_clean = (users_raw.filter(F.col("user_id").isNotNull()).dropDuplicates(["user_id"])
    .withColumn("registration_ts", F.to_timestamp("registration_date"))
    .withColumn("last_active_ts", F.to_timestamp("last_active"))
    .withColumn("city", F.initcap(F.trim(F.col("geo_data.city"))))
    .withColumn("state", F.upper(F.trim(F.col("geo_data.state"))))
    .withColumn("country", F.coalesce(F.upper(F.trim(F.col("geo_data.country"))), F.lit("UNKNOWN")))     # missing -> explicit UNKNOWN
    .withColumn("registration_month", F.date_format("registration_ts", "yyyy-MM"))
    .withColumn("dq_last_active_before_registration", F.col("last_active_ts") < F.col("registration_ts"))
    .select("user_id", "city", "state", "country", "registration_ts", "registration_month", "last_active_ts", "dq_last_active_before_registration"))
users_clean.show(5, truncate=30)

+-----------+------------------+-----+-------+-------------------+------------------+-------------------+----------------------------------+
|    user_id|              city|state|country|    registration_ts|registration_month|     last_active_ts|dq_last_active_before_registration|
+-----------+------------------+-----+-------+-------------------+------------------+-------------------+----------------------------------+
|user_000000|       Mcintoshton|   MO|     ML|2026-05-09 23:30:28|           2026-05|2026-08-05 03:55:57|                             false|
|user_000001|       Michaelstad|   MO|     DJ|2026-06-12 20:04:39|           2026-06|2026-07-22 20:03:37|                             false|
|user_000002|      Lake Makayla|   PA|     AU|2026-06-02 02:20:39|           2026-06|2026-09-15 02:51:04|                             false|
|user_000003|        Danielfort|   SC|     ME|2026-04-18 04:34:24|           2026-04|2026-08-16 09:04:24|                             false|
|user_000004|

### 4.3 Transactions
Business rules enforced: `total = subtotal − discount`; `subtotal = Σ line items`; status/payment text normalised; **orders without a session are kept** but flagged (`has_session = false`) – roughly half of all orders, found in notebook 00.

In [6]:
transactions_clean = (transactions_raw.filter(F.col("transaction_id").isNotNull() & F.col("user_id").isNotNull()).dropDuplicates(["transaction_id"])
    .withColumn("txn_ts", F.to_timestamp("timestamp"))
    .withColumn("txn_date", F.to_date("txn_ts"))
    .withColumn("status", F.lower(F.trim("status")))
    .withColumn("payment_method", F.lower(F.trim("payment_method")))
    .withColumn("session_id", F.when(F.trim("session_id") == "", None).otherwise(F.col("session_id")))
    .withColumn("has_session", F.col("session_id").isNotNull())
    .withColumn("n_line_items", F.size("items"))
    .withColumn("n_units", F.aggregate("items", F.lit(0), lambda a, x: a + x["quantity"]))
    .withColumn("calc_subtotal", F.aggregate("items", F.lit(0.0), lambda a, x: a + x["subtotal"]))
    .withColumn("discount", F.coalesce("discount", F.lit(0.0)))
    .withColumn("discount_rate", F.when(F.col("subtotal") > 0, F.round(F.col("discount") / F.col("subtotal"), 3)).otherwise(0.0))
    .withColumn("dq_total_mismatch", F.abs(F.col("subtotal") - F.col("discount") - F.col("total")) > 0.01)
    .withColumn("dq_subtotal_mismatch", F.abs(F.col("subtotal") - F.col("calc_subtotal")) > 0.01)
    .withColumn("is_valid", ~(F.col("dq_total_mismatch") | F.col("dq_subtotal_mismatch") | (F.col("total") < 0) | (F.col("n_line_items") == 0))))

# Line-item grain (one row per product per order) with discount allocated proportionally, margin and category attached
prod_dim = products_clean.select("product_id", "product_name", "category_id", "category_name", "avg_profit_margin")
transaction_items = (transactions_clean.filter("is_valid")
    .select("transaction_id", "user_id", "txn_ts", "txn_date", "session_id", "subtotal", "total", F.explode("items").alias("it"))
    .select("transaction_id", "user_id", "txn_ts", "txn_date", "session_id",
            F.col("it.product_id").alias("product_id"), F.col("it.quantity").alias("quantity"),
            F.col("it.unit_price").alias("unit_price"), F.col("it.subtotal").alias("item_subtotal"),
            F.when(F.col("subtotal") > 0, F.col("total") / F.col("subtotal")).otherwise(1.0).alias("net_factor"))
    .withColumn("item_net_revenue", F.round(F.col("item_subtotal") * F.col("net_factor"), 2))
    .drop("net_factor")
    .join(F.broadcast(prod_dim), "product_id", "left")
    .withColumn("est_gross_profit", F.round(F.col("item_net_revenue") * F.col("avg_profit_margin"), 2)))
transaction_items.show(5, truncate=25)

+----------+----------------+-----------+-------------------+----------+----------+--------+----------+-------------+----------------+-------------------------+-----------+-------------------------+-----------------+----------------+
|product_id|  transaction_id|    user_id|             txn_ts|  txn_date|session_id|quantity|unit_price|item_subtotal|item_net_revenue|             product_name|category_id|            category_name|avg_profit_margin|est_gross_profit|
+----------+----------------+-----------+-------------------+----------+----------+--------+----------+-------------+----------------+-------------------------+-----------+-------------------------+-----------------+----------------+
|prod_00577|txn_000057265dba|user_000734|2026-10-30 02:32:18|2026-10-30|      NULL|       1|    100.95|       100.95|          100.95|Cross-Group Contextual...|    cat_005|          Cameron-Parsons|            0.275|           27.76|
|prod_04222|txn_000057265dba|user_000734|2026-10-30 02:32:18|202

### 4.4 Sessions
Derived columns: calendar attributes, device fields, funnel flags (`has_cart`, `reached_cart_page`, `reached_checkout`), cart value, and – crucially – **`is_purchased`**: whether a transaction really exists for the session. The generator's `conversion_status` says "converted" for sessions whose order was never created (stock failure), so we **reconcile** it with the transactions table.

In [7]:
purchased_sessions = (transactions_clean.filter("is_valid AND has_session").select("session_id").distinct().withColumn("is_purchased", F.lit(True)))

sessions_clean = (sessions_raw.filter(F.col("session_id").isNotNull() & F.col("user_id").isNotNull()).dropDuplicates(["session_id"])
    .withColumn("start_ts", F.to_timestamp("start_time")).withColumn("end_ts", F.to_timestamp("end_time"))
    .filter((F.col("duration_seconds") > 0) & F.col("start_ts").isNotNull())
    .withColumn("session_date", F.to_date("start_ts"))
    .withColumn("hour", F.hour("start_ts"))
    .withColumn("day_of_week", F.date_format("start_ts", "E"))
    .withColumn("dow_num", F.dayofweek("start_ts"))
    .withColumn("device_type", F.coalesce(F.lower(F.col("device_profile.type")), F.lit("unknown")))
    .withColumn("os", F.col("device_profile.os")).withColumn("browser", F.col("device_profile.browser"))
    .withColumn("country", F.upper(F.col("geo_data.country")))
    .withColumn("referrer", F.coalesce(F.lower("referrer"), F.lit("unknown")))
    .withColumn("viewed_products", F.coalesce("viewed_products", F.array().cast("array<string>")))
    .withColumn("n_page_views", F.size("page_views"))
    .withColumn("n_products_viewed", F.size("viewed_products"))
    .withColumn("has_cart", F.size(F.map_keys("cart_contents")) > 0)
    .withColumn("cart_value", F.round(F.aggregate(F.map_values("cart_contents"), F.lit(0.0), lambda a, x: a + x["quantity"] * x["price"]), 2))
    .withColumn("viewed_product_page", F.exists("page_views", lambda x: x["page_type"] == "product_detail"))
    .withColumn("reached_cart_page", F.exists("page_views", lambda x: x["page_type"] == "cart"))
    .withColumn("reached_checkout", F.exists("page_views", lambda x: x["page_type"] == "checkout"))
    .withColumn("reached_confirmation", F.exists("page_views", lambda x: x["page_type"] == "confirmation"))
    .withColumnRenamed("conversion_status", "conversion_status_raw")
    .join(purchased_sessions, "session_id", "left")
    .withColumn("is_purchased", F.coalesce("is_purchased", F.lit(False)))
    .drop("geo_data", "device_profile"))
sessions_clean.drop("page_views", "cart_contents", "viewed_products").show(5, truncate=25)

+---------------+-----------+-------------------+-------------------+----------------+---------------------+-------------+-------------------+-------------------+------------+----+-----------+-------+-----------+-------+-------+-------+------------+-----------------+--------+----------+-------------------+-----------------+----------------+--------------------+------------+
|     session_id|    user_id|         start_time|           end_time|duration_seconds|conversion_status_raw|     referrer|           start_ts|             end_ts|session_date|hour|day_of_week|dow_num|device_type|     os|browser|country|n_page_views|n_products_viewed|has_cart|cart_value|viewed_product_page|reached_cart_page|reached_checkout|reached_confirmation|is_purchased|
+---------------+-----------+-------------------+-------------------+----------------+---------------------+-------------+-------------------+-------------------+------------+----+-----------+-------+-----------+-------+-------+-------+----------

### 4.5 Flatten the nested arrays into analysis-ready tables
* `page_views` – one row per page view (with its position `seq` in the journey)
* `cart_items` – one row per product in a session's cart (map → rows)

In [8]:
page_views = (sessions_clean.select("session_id", "user_id", "session_date", "device_type", "referrer", F.posexplode("page_views").alias("seq", "pv"))
    .select("session_id", "user_id", "session_date", "device_type", "referrer", "seq",
            F.to_timestamp(F.col("pv.timestamp")).alias("ts"), F.col("pv.page_type").alias("page_type"),
            F.col("pv.product_id").alias("product_id"), F.col("pv.category_id").alias("category_id"),
            F.col("pv.view_duration").alias("view_duration")))

cart_items = (sessions_clean.filter("has_cart").select("session_id", "user_id", "session_date", "is_purchased", F.explode("cart_contents").alias("product_id", "line"))
    .select("session_id", "user_id", "session_date", "is_purchased", "product_id", F.col("line.quantity").alias("quantity"), F.col("line.price").alias("price")))
page_views.show(5, truncate=25); cart_items.show(5)

+---------------+-----------+------------+-----------+-------------+---+-------------------+----------------+----------+-----------+-------------+
|     session_id|    user_id|session_date|device_type|     referrer|seq|                 ts|       page_type|product_id|category_id|view_duration|
+---------------+-----------+------------+-----------+-------------+---+-------------------+----------------+----------+-----------+-------------+
|sess_000936dffc|user_006477|  2026-11-06|     tablet|search_engine|  0|2026-11-06 03:59:40|            home|      NULL|       NULL|           89|
|sess_000936dffc|user_006477|  2026-11-06|     tablet|search_engine|  1|2026-11-06 04:01:09|          search|      NULL|       NULL|          928|
|sess_000936dffc|user_006477|  2026-11-06|     tablet|search_engine|  2|2026-11-06 04:16:37|  product_detail|prod_03350|    cat_008|          199|
|sess_000936dffc|user_006477|  2026-11-06|     tablet|search_engine|  3|2026-11-06 04:19:56|category_listing|      NUL

### 4.6 Data-quality report (raw vs clean, nulls, rule violations)
One Spark action per table collects all counts at once. The result is saved for the report.

In [9]:
def counts(df, **conds):
    row = df.agg(F.count("*").alias("rows"), *[F.sum(F.when(c, 1).otherwise(0)).alias(k) for k, c in conds.items()]).first().asDict()
    return {k: int(v or 0) for k, v in row.items()}

dq = []
def add(table, check, bad, total, action):
    dq.append({"table": table, "check": check, "violations": bad, "of_rows": total, "action": action})

raw_n = {"users": users_raw.count(), "products": products_raw.count(), "transactions": transactions_raw.count(), "sessions": sessions_raw.count()}
add("users", "rows dropped by cleaning (null key / duplicate)", raw_n["users"] - users_clean.count(), raw_n["users"], "removed")
add("products", "rows dropped by cleaning", raw_n["products"] - products_clean.count(), raw_n["products"], "removed")

c = counts(transactions_clean, total_mismatch=F.col("dq_total_mismatch"), subtotal_mismatch=F.col("dq_subtotal_mismatch"),
           invalid=~F.col("is_valid"), no_session=~F.col("has_session"), null_user=F.col("user_id").isNull())
add("transactions", "rows dropped by cleaning (null key / duplicate)", raw_n["transactions"] - c["rows"], raw_n["transactions"], "removed")
add("transactions", "total != subtotal - discount", c["total_mismatch"], c["rows"], "flagged + excluded")
add("transactions", "subtotal != sum(line items)", c["subtotal_mismatch"], c["rows"], "flagged + excluded")
add("transactions", "no linked session (session_id is null)", c["no_session"], c["rows"], "kept, flagged has_session=false")

c = counts(sessions_clean, converted_raw=F.col("conversion_status_raw") == "converted",
           converted_raw_but_no_txn=(F.col("conversion_status_raw") == "converted") & ~F.col("is_purchased"),
           purchased_but_not_converted_raw=(F.col("conversion_status_raw") != "converted") & F.col("is_purchased"),
           cart_but_no_product_view=F.col("has_cart") & ~F.col("viewed_product_page"),
           no_pageviews=F.col("n_page_views") == 0)
add("sessions", "rows dropped by cleaning (null key / duplicate / duration<=0)", raw_n["sessions"] - c["rows"], raw_n["sessions"], "removed")
add("sessions", "conversion_status='converted' but NO transaction", c["converted_raw_but_no_txn"], c["converted_raw"], "reconciled -> is_purchased")
add("sessions", "has transaction but status != converted", c["purchased_but_not_converted_raw"], c["rows"], "reconciled -> is_purchased")
add("sessions", "cart without product-detail view", c["cart_but_no_product_view"], c["rows"], "none expected")

dq_df = pd.DataFrame(dq); dq_df["pct"] = (dq_df.violations / dq_df.of_rows * 100).round(2)
dq_df.to_csv(TAB / "data_quality_report.csv", index=False); display(dq_df)

,table,check,violations,of_rows,action,pct
0,users,rows dropped by cleaning (null key / duplicate),0,10000,removed,0.00
1,products,rows dropped by cleaning,0,5000,removed,0.00
2,transactions,rows dropped by cleaning (null key / duplicate),0,50000,removed,0.00
3,transactions,total != subtotal - discount,0,50000,flagged + excluded,0.00
4,transactions,subtotal != sum(line items),0,50000,flagged + excluded,0.00
5,transactions,no linked session (session_id is null),24906,50000,"kept, flagged has_session=false",49.81
6,sessions,rows dropped by cleaning (null key / duplicate...,0,200000,removed,0.00
7,sessions,conversion_status='converted' but NO transaction,15515,40609,reconciled -> is_purchased,38.21
8,sessions,has transaction but status != converted,0,200000,reconciled -> is_purchased,0.00
9,sessions,cart without product-detail view,0,200000,none expected,0.00


## Step 5 · Write the clean tables as Parquet
* **Parquet** = columnar + compressed + keeps the schema → later notebooks read only the columns they need and never re-parse JSON.
* **`sessions`, `page_views`: partitioned by `session_date`** → a query for one week reads ~7 of ~90 folders (*partition pruning*).
* `repartition("session_date")` first → exactly **one file per day** instead of hundreds of tiny files (the classic small-files problem).

In [10]:
OUT = PROCESSED / "parquet"
if OUT.exists(): shutil.rmtree(OUT)

def write(df, name, partition_by=None):
    w = df
    if partition_by: w = w.repartition(partition_by)
    wr = w.write.mode("overwrite")
    if partition_by: wr = wr.partitionBy(partition_by)
    with timer(f"write {name}"): wr.parquet(str(OUT / name))

write(categories_clean, "categories"); write(subcategories_dim, "subcategories"); write(products_clean, "products"); write(users_clean, "users")
write(transactions_clean, "transactions"); write(transaction_items, "transaction_items")
write(sessions_clean, "sessions", "session_date"); write(page_views, "page_views", "session_date"); write(cart_items, "cart_items", "session_date")

[timer] write categories: 1.92s
[timer] write subcategories: 0.18s
[timer] write products: 1.47s
[timer] write users: 1.22s
[timer] write transactions: 4.29s
[timer] write transaction_items: 4.32s
[timer] write sessions: 23.08s
[timer] write page_views: 13.63s
[timer] write cart_items: 7.96s


In [11]:
# From here on we read ONLY the Parquet output - the pipeline's contract with the other notebooks
rd = lambda n: spark.read.parquet(str(OUT / n))
sessions, txns, items, users, products = rd("sessions"), rd("transactions"), rd("transaction_items"), rd("users"), rd("products")
print({n: rd(n).count() for n in ["users", "products", "categories", "transactions", "transaction_items", "sessions", "page_views", "cart_items"]})

{'users': 10000, 'products': 5000, 'categories': 25, 'transactions': 50000, 'transaction_items': 109863, 'sessions': 200000, 'page_views': 1998578, 'cart_items': 208004}


## Step 6 · Product recommendation indicators
Association-rule metrics for a pair of products (A, B), computed over *baskets* (orders or sessions):

| Metric | Meaning |
|---|---|
| **pair_count** | baskets containing both A and B |
| **support** | pair_count / all baskets |
| **confidence (A→B)** | P(B \| A) = pair_count / baskets with A |
| **lift** | confidence / P(B) – **> 1** means A and B appear together *more than chance*; **≈ 1** = independent |

Implementation: a **self-join on the basket id** with `a.product < b.product` (each pair once, no (A,A)), then a `groupBy` – one shuffle for the join, one for the aggregation.
We do **both** variants the assignment lists: *"bought together"* (transactions) and *"viewed together"* (sessions).

In [12]:
def pair_metrics(basket_items, basket_col, min_count=2):
    """basket_items: DataFrame(basket_col, product_id) with DISTINCT rows. Returns pair metrics with lift."""
    n_baskets = basket_items.select(basket_col).distinct().count()
    single = basket_items.groupBy("product_id").agg(F.countDistinct(basket_col).alias("n"))
    a, b = basket_items.alias("a"), basket_items.alias("b")
    pairs = (a.join(b, (F.col(f"a.{basket_col}") == F.col(f"b.{basket_col}")) & (F.col("a.product_id") < F.col("b.product_id")))
               .groupBy(F.col("a.product_id").alias("product_a"), F.col("b.product_id").alias("product_b"))
               .agg(F.count("*").alias("pair_count"))
               .filter(F.col("pair_count") >= min_count))
    return (pairs.join(single.select(F.col("product_id").alias("product_a"), F.col("n").alias("n_a")), "product_a")
                 .join(single.select(F.col("product_id").alias("product_b"), F.col("n").alias("n_b")), "product_b")
                 .withColumn("support", F.col("pair_count") / n_baskets)
                 .withColumn("confidence_a_to_b", F.col("pair_count") / F.col("n_a"))
                 .withColumn("confidence_b_to_a", F.col("pair_count") / F.col("n_b"))
                 .withColumn("lift", F.col("confidence_a_to_b") / (F.col("n_b") / n_baskets))
                 .select("product_a", "product_b", "pair_count", "n_a", "n_b", "support", "confidence_a_to_b", "confidence_b_to_a", "lift")), n_baskets

pname = products.select("product_id", "product_name")
def with_names(df):
    return (df.join(pname.withColumnRenamed("product_id", "product_a").withColumnRenamed("product_name", "name_a"), "product_a")
              .join(pname.withColumnRenamed("product_id", "product_b").withColumnRenamed("product_name", "name_b"), "product_b"))

### 6.1 "Customers who bought X also bought Y" (transactions)

In [13]:
basket_txn = items.select("transaction_id", "product_id").distinct()
with timer("co-purchase pairs"):
    copurchase, n_orders = pair_metrics(basket_txn, "transaction_id", min_count=2)
    copurchase = copurchase.cache(); n_pairs = copurchase.count()
print(f"{n_orders:,} orders -> {n_pairs:,} product pairs bought together at least twice")
top_copurchase = with_names(copurchase).orderBy(F.desc("pair_count"), F.desc("lift")).limit(20).toPandas()
display(top_copurchase[["product_a", "name_a", "product_b", "name_b", "pair_count", "confidence_a_to_b", "lift"]].round(3).head(10))
top_copurchase.to_csv(TAB / "spark_copurchase_top.csv", index=False)

[timer] co-purchase pairs: 6.22s
50,000 orders -> 351 product pairs bought together at least twice


,product_a,name_a,product_b,name_b,pair_count,confidence_a_to_b,lift
0,prod_01405,Monitored 5Thgeneration Circuit,prod_03829,User-Centric Value-Added Framework,3,0.143,264.550
1,prod_03404,Robust Value-Added Adapter,prod_04408,Switchable Explicit Neural-Net,2,0.125,520.833
2,prod_00592,Devolved Exuding Graphic Interface,prod_00838,Grass-Roots Multi-State Knowledgebase,2,0.083,416.667
3,prod_02578,Universal Intangible Moderator,prod_04320,Phased Actuating Installation,2,0.111,370.370
4,prod_00515,Face-To-Face Exuding Strategy,prod_01400,Front-Line Didactic Parallelism,2,0.118,367.647
5,prod_03768,Synergized Regional Interface,prod_04246,Public-Key Full-Range Methodology,2,0.087,362.319
6,prod_00136,Secured Web-Enabled Methodology,prod_03982,Persevering Reciprocal Concept,2,0.118,326.797
7,prod_00922,Multi-Tiered Bottom-Line Neural-Net,prod_04507,Self-Enabling Asymmetric Project,2,0.111,326.797
8,prod_02121,Vision-Oriented Radical Process Improvement,prod_02169,Pre-Emptive Demand-Driven Superstructure,2,0.087,310.559
9,prod_02675,Multi-Channeled Heuristic Moratorium,prod_04435,Open-Architected Needs-Based Collaboration,2,0.087,310.559


### 6.2 "Products frequently viewed together" (sessions)

In [14]:
basket_view = (sessions.filter(F.size("viewed_products") >= 2).select("session_id", F.explode("viewed_products").alias("product_id")).distinct())
with timer("co-view pairs"):
    coview, n_sess = pair_metrics(basket_view, "session_id", min_count=3)
    coview = coview.cache(); n_vpairs = coview.count()
print(f"{n_sess:,} multi-product sessions -> {n_vpairs:,} product pairs viewed together at least 3 times")
top_coview = with_names(coview).orderBy(F.desc("pair_count"), F.desc("lift")).limit(20).toPandas()
display(top_coview[["product_a", "name_a", "product_b", "name_b", "pair_count", "confidence_a_to_b", "lift"]].round(3).head(10))
top_coview.to_csv(TAB / "spark_coview_top.csv", index=False)

[timer] co-view pairs: 13.08s
171,920 multi-product sessions -> 2,130 product pairs viewed together at least 3 times


,product_a,name_a,product_b,name_b,pair_count,confidence_a_to_b,lift
0,prod_02212,Automated System-Worthy Portal,prod_03015,Compatible Clear-Thinking Migration,5,0.034,39.511
1,prod_03440,Focused Tangible Artificial Intelligence,prod_03947,Programmable Reciprocal Product,5,0.031,38.451
2,prod_02483,Polarized Explicit Forecast,prod_02924,Polarized Clear-Thinking Flexibility,4,0.029,43.088
3,prod_01863,Exclusive Responsive Function,prod_02752,Public-Key Multi-Tasking Analyzer,4,0.033,40.552
4,prod_01474,Re-Engineered Clear-Thinking Migration,prod_04409,Virtual Radical Synergy,4,0.028,40.471
5,prod_01005,Reverse-Engineered Needs-Based Support,prod_02546,Devolved Upward-Trending System Engine,4,0.026,39.427
6,prod_03847,Team-Oriented Next Generation Flexibility,prod_04694,Grass-Roots Well-Modulated Array,4,0.029,38.911
7,prod_02797,Object-Based Bandwidth-Monitored Definition,prod_03819,Profound Composite Infrastructure,4,0.029,37.751
8,prod_00148,Up-Sized Exuding Parallelism,prod_04044,Reactive Mission-Critical Analyzer,4,0.028,37.603
9,prod_02555,Balanced Bi-Directional Moratorium,prod_03819,Profound Composite Infrastructure,4,0.029,37.480


### Is the affinity *real*? – a null-model check
Careful: lift of the pairs we keep is **biased upwards**. We only keep pairs seen ≥ 3 times, but with thousands of products, chance alone makes most pairs co-occur 0–1 times, so *every survivor has a high lift by construction*. A large lift therefore proves nothing on its own.

A fair test: build a **null model** – the *same baskets* (same sizes) but with products re-assigned **at random** – and run the *identical* pipeline. If real data shows no more (or stronger) pairs than the null, there is no affinity beyond chance.

In [15]:
n_prod = products.count()
pidx = products.select("product_id").withColumn("rn", F.row_number().over(Window.orderBy("product_id")))
null_view = (basket_view.drop("product_id").withColumn("rn", (F.floor(F.rand(7) * n_prod) + 1).cast("int"))   # same baskets, products re-drawn at random
             .join(F.broadcast(pidx), "rn").select("session_id", "product_id").distinct())
with timer("null-model co-view pairs"):
    null_pairs, _ = pair_metrics(null_view, "session_id", min_count=3)
    null_pairs = null_pairs.cache()

def summarise(df, label):
    p = df.select("pair_count", "lift").toPandas()
    return {"model": label, "pairs_kept": len(p), "median_lift": round(p.lift.median(), 2),
            "p95_pair_count": int(p.pair_count.quantile(.95)), "max_pair_count": int(p.pair_count.max())}
cmp_df = pd.DataFrame([summarise(coview, "REAL sessions"), summarise(null_pairs, "NULL (random products)")])
display(cmp_df); cmp_df.to_csv(TAB / "spark_coview_vs_null.csv", index=False)

[timer] null-model co-view pairs: 5.25s


,model,pairs_kept,median_lift,p95_pair_count,max_pair_count
0,REAL sessions,2130,24.66,3,5
1,NULL (random products),1652,28.28,3,4


> **Reading the result.** If the two rows are close (same order of magnitude of pairs, similar median lift and maximum count), the "recommendations" are **statistically indistinguishable from chance**. That is what we expect: the generator draws viewed products at random, so there is no true affinity to find.
> This is a **limitation / critical-thinking** point for the report: the *pipeline* (self-join → metrics → window ranking) is correct and will surface real affinity on real data, but on this synthetic data the top pairs are noise – a naive reading of "lift = 50" would be wrong. In production you would also raise `min_count`, add a significance test and validate recommendations on a hold-out period.

### 6.3 Turn the pairs into a "top-5 recommendations per product" table
Make the relation symmetric (A→B and B→A), rank per source product with a **window function**, keep 5.

In [16]:
edges = (coview.select(F.col("product_a").alias("source"), F.col("product_b").alias("recommended"), "pair_count", F.col("confidence_a_to_b").alias("confidence"), "lift")
         .unionByName(coview.select(F.col("product_b").alias("source"), F.col("product_a").alias("recommended"), "pair_count", F.col("confidence_b_to_a").alias("confidence"), "lift")))
w = Window.partitionBy("source").orderBy(F.desc("pair_count"), F.desc("lift"))
recs = edges.withColumn("rank", F.row_number().over(w)).filter("rank <= 5")
recs.write.mode("overwrite").parquet(str(OUT / "recommendations_coview"))
sample_src = recs.groupBy("source").count().filter("count = 5").first()["source"]
print("Recommendations for", sample_src); recs.filter(F.col("source") == sample_src).orderBy("rank").show(truncate=False)

Recommendations for prod_01144
+----------+-----------+----------+--------------------+------------------+----+
|source    |recommended|pair_count|confidence          |lift              |rank|
+----------+-----------+----------+--------------------+------------------+----+
|prod_01144|prod_03193 |3         |0.018867924528301886|24.38927507447865 |1   |
|prod_01144|prod_01392 |3         |0.018867924528301886|23.677179451866134|2   |
|prod_01144|prod_04424 |3         |0.018867924528301886|23.00548641777064 |3   |
|prod_01144|prod_01784 |3         |0.018867924528301886|22.683731362976644|4   |
|prod_01144|prod_03062 |3         |0.018867924528301886|20.927571515520388|5   |
+----------+-----------+----------+--------------------+------------------+----+



## Step 7 · Cohort analysis (registration month → spending over time)
* **Cohort** = users grouped by **registration month**.
* **Age of a purchase** = `floor(months_between(order time, registration time))`.
* For each (cohort, age): **revenue**, **active buyers**, **revenue per cohort user** and **retention** (= buyers ÷ cohort size).

> Because the generator registers users 3–9 months *before* the 90-day activity window, the cohort ages we can observe start at ~3 months – an artefact to mention in the report.

In [17]:
cohort_size = users.groupBy("registration_month").agg(F.count("*").alias("cohort_users"))
enriched = (txns.filter("is_valid").join(users.select("user_id", "registration_ts", "registration_month"), "user_id")
              .withColumn("months_since_reg", F.floor(F.months_between("txn_ts", "registration_ts")).cast("int")))

cohort = (enriched.groupBy("registration_month", "months_since_reg")
          .agg(F.round(F.sum("total"), 2).alias("revenue"), F.countDistinct("user_id").alias("active_buyers"), F.count("*").alias("orders"))
          .join(cohort_size, "registration_month")
          .withColumn("revenue_per_user", F.round(F.col("revenue") / F.col("cohort_users"), 2))
          .withColumn("retention_pct", F.round(F.col("active_buyers") / F.col("cohort_users") * 100, 1))
          .orderBy("registration_month", "months_since_reg"))
cohort_pd = cohort.toPandas(); cohort_pd.to_csv(TAB / "spark_cohort.csv", index=False)

print("Revenue per cohort user (rows = registration month, columns = months since registration)")
display(cohort_pd.pivot(index="registration_month", columns="months_since_reg", values="revenue_per_user").round(2))
print("Retention % (buyers / cohort size)")
display(cohort_pd.pivot(index="registration_month", columns="months_since_reg", values="retention_pct"))

Revenue per cohort user (rows = registration month, columns = months since registration)


months_since_reg,0,1,2,3,4,5,6,7,8
registration_month,,,,,,,,,
2026-02,NaN,NaN,NaN,NaN,NaN,521.80,1879.28,1839.46,1275.93
2026-03,NaN,NaN,NaN,NaN,387.87,1772.24,1806.26,1360.35,71.52
2026-04,NaN,NaN,NaN,341.34,1754.21,1812.72,1403.31,78.38,NaN
2026-05,NaN,NaN,364.21,1773.16,1864.07,1363.76,74.28,NaN,NaN
2026-06,NaN,379.16,1715.83,1822.08,1386.36,76.92,NaN,NaN,NaN
2026-07,405.34,1700.13,1740.99,1374.36,74.42,NaN,NaN,NaN,NaN
2026-08,1523.00,1780.39,1825.42,237.30,NaN,NaN,NaN,NaN,NaN


Retention % (buyers / cohort size)


months_since_reg,0,1,2,3,4,5,6,7,8
registration_month,,,,,,,,,
2026-02,NaN,NaN,NaN,NaN,NaN,36.5,81.0,80.9,67.1
2026-03,NaN,NaN,NaN,NaN,25.5,81.1,81.6,71.0,6.1
2026-04,NaN,NaN,NaN,23.5,79.8,81.4,69.3,5.9,NaN
2026-05,NaN,NaN,25.9,80.5,81.1,68.7,5.8,NaN,NaN
2026-06,NaN,25.7,79.4,80.4,70.9,5.6,NaN,NaN,NaN
2026-07,27.5,79.5,81.9,70.5,5.8,NaN,NaN,NaN,NaN
2026-08,74.8,79.1,82.5,19.2,NaN,NaN,NaN,NaN,NaN


### 7.1 Exposure-fair cohort view (equal 30-day periods)
**Caveat on the table above:** the 90-day activity window cuts the "months since registration" buckets unevenly – the first and last bucket of each cohort are only *partially observed*, so their revenue and retention look artificially low. Comparing bucket 5 with bucket 6 would be comparing a 10-day slice with a 30-day slice.

Fix: cut the observation window into **three equal 30-day periods** (P1 oldest … P3 newest) and measure each registration-month cohort in each period. Every cell has the same exposure, so cohorts and periods can be compared directly.

In [18]:
win_start = sessions.agg(F.min("session_date")).first()[0]
print("Observation window starts:", win_start)
period = (txns.filter("is_valid").join(users.select("user_id", "registration_month"), "user_id")
          .withColumn("period", F.least(F.floor(F.datediff("txn_date", F.lit(win_start)) / 30), F.lit(2)).cast("int") + 1)
          .groupBy("registration_month", "period")
          .agg(F.round(F.sum("total"), 2).alias("revenue"), F.countDistinct("user_id").alias("active_buyers"), F.count("*").alias("orders"))
          .join(cohort_size, "registration_month")
          .withColumn("revenue_per_user", F.round(F.col("revenue") / F.col("cohort_users"), 2))
          .withColumn("buyer_rate_pct", F.round(F.col("active_buyers") / F.col("cohort_users") * 100, 1)))
period_pd = period.toPandas().sort_values(["registration_month", "period"]); period_pd.to_csv(TAB / "spark_cohort_periods.csv", index=False)
print("Revenue per cohort user, by equal 30-day period (P1 = oldest)")
display(period_pd.pivot(index="registration_month", columns="period", values="revenue_per_user").round(2))
print("% of cohort that bought in the period")
display(period_pd.pivot(index="registration_month", columns="period", values="buyer_rate_pct"))

Observation window starts: 2026-08-11
Revenue per cohort user, by equal 30-day period (P1 = oldest)


period,1,2,3
registration_month,,,
2026-02,1799.21,1837.80,1879.46
2026-03,1810.34,1802.60,1785.30
2026-04,1789.69,1753.80,1846.47
2026-05,1761.82,1862.61,1815.05
2026-06,1761.42,1788.94,1829.99
2026-07,1687.98,1802.06,1805.19
2026-08,1746.60,1796.96,1822.56


% of cohort that bought in the period


period,1,2,3
registration_month,,,
2026-02,79.8,80.3,83.3
2026-03,82.0,81.6,81.0
2026-04,80.9,79.9,81.5
2026-05,80.8,80.9,80.7
2026-06,80.1,81.0,82.3
2026-07,79.3,81.7,82.5
2026-08,80.1,81.8,83.0


## Step 8 · Optimisation evidence (numbers for the report)
Four choices were made above; here we *measure* them.

In [19]:
# (a) multi-line JSON array vs NDJSON vs Parquet - same aggregation: sessions per device.
# Each variant runs twice: run 1 includes JVM / file-listing warm-up, run 2 is the fair comparison.
def device_count(df):
    col = "device_profile.type" if "device_profile" in df.columns else "device_type"
    return df.groupBy(col).count().collect()

def bench(label, make_df):
    for run in (1, 2):
        with timer(f"{label} (run {run})"):
            device_count(make_df())

bench("(a1) JSON ARRAY files, multiLine=true", lambda: spark.read.schema(sessions_schema).option("multiLine", True).json(str(RAW / "sessions_*.json")))
bench("(a2) NDJSON, splittable             ", lambda: spark.read.schema(sessions_schema).json(str(NDJSON / "sessions")))
bench("(a3) Parquet, 1 column read         ", lambda: spark.read.parquet(str(OUT / "sessions")))
print("NOTE: at SCALE=small the files are tiny and fixed overhead dominates - the Parquet advantage appears at SCALE=medium and grows with size. Quote the medium numbers.")

[timer] (a1) JSON ARRAY files, multiLine=true (run 1): 2.09s
[timer] (a1) JSON ARRAY files, multiLine=true (run 2): 1.61s
[timer] (a2) NDJSON, splittable              (run 1): 3.81s
[timer] (a2) NDJSON, splittable              (run 2): 3.84s
[timer] (a3) Parquet, 1 column read          (run 1): 4.15s
[timer] (a3) Parquet, 1 column read          (run 2): 1.06s
NOTE: at SCALE=small the files are tiny and fixed overhead dominates - the Parquet advantage appears at SCALE=medium and grows with size. Quote the medium numbers.


In [20]:
# (b) Partition pruning: one week of sessions reads only 7 date folders
wk = spark.read.parquet(str(OUT / "sessions")).filter(F.col("session_date").between(F.date_sub(F.current_date(), 7), F.current_date()))
wk.agg(F.count("*")).explain()          # look for: PartitionFilters: [... session_date ...]

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[], functions=[count(1)])
   +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=13003]
      +- HashAggregate(keys=[], functions=[partial_count(1)])
         +- Project
            +- FileScan parquet [session_date#6922] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/E:/bigdata/auca-ecommerce-analytics/data/processed/parquet/sessi..., PartitionFilters: [isnotnull(session_date#6922), (session_date#6922 >= 2026-11-02), (session_date#6922 <= 2026-11-09)], PushedFilters: [], ReadSchema: struct<>




In [21]:
# (c) Broadcast join of the small dimension: look for BroadcastHashJoin in the plan (no shuffle of the big side)
items.join(F.broadcast(products.select("product_id", "current_price")), "product_id").groupBy("category_id").count().explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[category_id#2763], functions=[count(1)])
   +- Exchange hashpartitioning(category_id#2763, 16), ENSURE_REQUIREMENTS, [plan_id=13045]
      +- HashAggregate(keys=[category_id#2763], functions=[partial_count(1)])
         +- Project [category_id#2763]
            +- BroadcastHashJoin [product_id#2752], [product_id#2775], Inner, BuildRight, false
               :- Filter isnotnull(product_id#2752)
               :  +- FileScan parquet [product_id#2752,category_id#2763] Batched: true, DataFilters: [isnotnull(product_id#2752)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/E:/bigdata/auca-ecommerce-analytics/data/processed/parquet/trans..., PartitionFilters: [], PushedFilters: [IsNotNull(product_id)], ReadSchema: struct<product_id:string,category_id:string>
               +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, false]),false), [plan_id=13040]
                  +- Filte

**Optimisation choices to describe in the report**

| Choice | Effect |
|---|---|
| Explicit schemas (no inference) | saves a full data pass; avoids a 5,000-column struct from `cart_contents`; `MapType` keeps it compact |
| JSON array → NDJSON | splittable input → all cores work in parallel, bounded memory |
| Parquet + column pruning | analytics read 1–5 of ~25 columns; typically 5–10× faster than JSON |
| `partitionBy(session_date)` + `repartition` | partition pruning; one file per day (no small-files problem) |
| Broadcast joins for `products` / `categories` | dimension tables are tiny → the large side is never shuffled |
| `spark.sql.shuffle.partitions = 16` (default 200) | 200 tiny tasks on one laptop is pure overhead |
| `cache()` of `copurchase` / `coview` | reused several times (top-N, lift stats, recommendations) |
| Distinct baskets before the self-join | join output size is quadratic in basket size – dedupe first |

**Scalability:** none of the code assumes the data fits in one machine. On a cluster you would change only the master URL and the input path (`hdfs://` / `s3://`), raise shuffle partitions to ~2–3× total cores, and keep Parquet partitioning by date.

In [22]:
spark.stop()
print("Parquet written to:", OUT)

Parquet written to: E:\bigdata\auca-ecommerce-analytics\data\processed\parquet


### ✅ What you should have after this notebook
* `data/processed/parquet/` with: `users, products, categories, subcategories, transactions, transaction_items, sessions, page_views, cart_items, recommendations_coview`.
* `outputs/tables/`: `data_quality_report`, `spark_copurchase_top`, `spark_coview_top`, `spark_coview_vs_null`, `spark_cohort`, `spark_cohort_periods`.
* Timing numbers from Step 8.

➡ Next: `04_spark_sql_analytics.ipynb`.